NOTE: A similar validation is performed in `test_model.ipynb`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from typing import Literal
import copy

from phd_visualizations.regression import regression_plot
from phd_visualizations.calculations import SupportedInstruments 
from phd_visualizations import save_figure

import combined_cooler
import matlab

cc_model = combined_cooler.initialize()
print(f"Using MATLAB package: {cc_model.name}")

%reload_ext autoreload
%autoreload 2

data_path = Path("../data")
model_type: Literal["physical", "data"] = "data"

df = pd.read_csv(data_path / "cc_out_exp.csv")
df["Tcc_out"] = df["Tc_in"]


display(
    df.loc[(df["qdc"] < 5) | (df["wdc"] < 11), ["qdc", "Tdc_in", "Tdc_out", "wdc"]]
)
display(
    df.loc[(df["Tv"] > 45)]
)
display(
    df.loc[(df["qwct"] < 6)]
)


,qdc,Tdc_in,Tdc_out,wdc
2,0.235675,21.977082,20.147901,0.0
3,0.390683,31.971167,23.391675,0.0
6,0.119232,25.526867,25.948971,0.0
14,0.119232,25.526867,25.948971,0.0
16,0.283769,16.941505,16.312816,0.0
22,0.000544,42.955688,38.732794,0.0


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,mv,Tdc_out,Twct_out,wdc,wwct,qdc,qwct,Rp,Rs,Tcc_out
15,27-May-2025,45.425107,29.240422,17.713841,23.242701,36.217052,0.335793,5.182903,44.957641,42.762154,...,245.835179,40.941588,36.052328,20.003386,28.648481,23.044723,22.908836,0.009,0.986,36.217052
17,24-Jul-2025,56.215542,26.423901,56.543519,24.000492,47.035182,0.211533,3.365881,55.892954,53.416242,...,262.622159,50.502149,39.505497,14.625326,21.000308,16.000408,7.770829,0.333,0.000,47.035182
20,01-Jul-2025,54.648207,36.677142,14.316505,24.000688,44.328101,0.161654,4.727527,53.423163,50.083774,...,239.540017,49.305879,42.593710,11.003121,21.000000,23.789244,18.207007,0.009,0.756,44.328101
21,03-Jul-2025,56.608496,33.842684,31.552103,24.000054,48.105297,0.342105,3.646723,56.431734,54.263118,...,274.686934,50.445927,40.643478,26.442129,21.000000,18.000156,5.927470,0.250,0.000,48.105297
22,27-Jun-2025,54.674012,32.536467,21.397523,23.999274,45.564017,0.128667,4.527399,54.001999,42.955688,...,229.925265,38.732794,45.397477,0.000000,21.000565,0.000544,23.718957,1.000,0.000,45.564017
23,11-Jul-2025,54.362551,27.937277,31.445756,23.999951,44.114168,0.161603,3.791804,54.053380,51.311346,...,298.851486,46.448266,43.052616,11.000000,21.000000,6.000635,17.747227,0.750,0.000,44.114168


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,mv,Tdc_out,Twct_out,wdc,wwct,qdc,qwct,Rp,Rs,Tcc_out
8,13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,...,214.037281,36.973783,15.001290,30.000000,0.0,23.798313,0.00054,0.008,0.0,37.228958
21,03-Jul-2025,56.608496,33.842684,31.552103,24.000054,48.105297,0.342105,3.646723,56.431734,54.263118,...,274.686934,50.445927,40.643478,26.442129,21.0,18.000156,5.92747,0.250,0.0,48.105297


### Evaluate model

In [ ]:
parameters_default = cc_model.default_parameters(nargout=1)

results_dfs = []
for model_type in ["data", "physical"]:
    options = {
        "model_type": model_type,  # "data" or "physical"
        # DC                     "Tamb",   "Tin"  "q", "w_fan"
        "dc_lb": matlab.double([[5.0600,   10.0,  5.0, 11]]),
        "dc_ub": matlab.double([[50.7500,   55.0, 24.5, 99.1800]]),
        "dc_model_data_path": "/workspaces/SOLhycool/modeling/data/models_data/model_data_dc_fp_pilot_plant_gaussian_cascade.mat",
        # WCT                    "Tamb",  "HR",   "Tin",  "q",       "w_fan"
        "wct_lb": matlab.double([[0.1,    0.1,     5.0,    5.0,       0.]]),
        "wct_ub": matlab.double([[50.0,   99.99,   55.0,   24.5,      95.]]),
        "wct_model_data_path": "/workspaces/SOLhycool/modeling/data/models_data/model_data_wct_fp_pilot_plant_gaussian_cascade.mat",
    }

    results: list[dict] = []
    for idx, ds in df.iterrows():

        qc_m3h = matlab.double([ds["qc"]])
        qdc_m3h = matlab.double([ds["qdc"]])
        qwct_m3h = matlab.double([ds["qwct"]])

        Rp, Rs = cc_model.flows_to_ratios(qc_m3h, qdc_m3h, qwct_m3h, nargout=2)

        Tamb_C = matlab.double([ds["Tamb"]])
        HR_pp = matlab.double([ds["HR"]])
        mv_kgh = matlab.double([ds["mv"]])
        wdc = matlab.double([ds["wdc"]])
        wwct = matlab.double([ds["wwct"]])
        Tv = matlab.double([ds["Tv"]])

        # Create the 'options' struct as a Python dictionary
        Ce_kWe, Cw_lh, detailed = cc_model.combined_cooler_model(Tamb_C, HR_pp, mv_kgh, qc_m3h, Rp, Rs, wdc, wwct, Tv, options, nargout=3)
        
        results.append(detailed)
        
    results_dfs.append( pd.DataFrame(results) )

display(results_dfs[0])
display(results_dfs[1])


> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In combined_cooler_model (line 191)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In combined_cooler_model (line 191)
> In wct_model_data (line 80)
In combined_cooler_model (line 172)
> In combined_cooler_model (line 191)
> In combined_cooler_model (line 191)
> In combined_cooler_model (line 191)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In combined_cooler_model (line 191)
> In combined_cooler_model (line 191)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In dc_model_data (line 71)
In combined_cooler_model (line 155)
> In combined_cool

,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,5.096935,99.564705,...,33.159526,28.074970,0.312366,73.241613,12.261262,32.881692,27.193027,0.138014,99.564705,80.982743
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,5.079493,109.410400,...,33.821712,27.190805,0.284241,92.383495,12.000188,33.821712,26.742242,0.147773,109.410400,98.634443
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,5.028469,208.352229,...,33.278941,33.278941,0.000000,0.000000,23.759322,33.278941,27.460768,0.381511,208.352229,160.493671
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,5.313815,197.584734,...,33.888814,33.888814,0.000000,0.000000,23.691455,33.888814,27.308512,0.667380,197.584734,180.995768
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,11.755116,128.633147,...,33.771250,29.678827,5.867400,66.475210,10.007950,33.771250,23.957540,1.240480,128.633147,114.041362
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,12.511319,165.978812,...,34.989055,30.866282,5.867400,66.968833,10.008669,34.989055,22.718555,1.995837,165.978812,142.601130
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,5.030727,233.367627,...,40.385228,40.385228,0.000000,0.000000,23.880049,40.385228,34.322757,0.383408,233.367627,168.033483
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,12.810758,205.878806,...,35.843477,32.151788,5.867400,24.374000,18.311492,35.843477,27.371303,2.296391,205.878806,180.103848
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,4.920243,0.000000,...,42.299130,36.812162,0.271859,151.633278,0.192018,42.299130,42.299130,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,4.204095,136.001713,...,39.959556,29.958563,1.791859,136.646022,6.227499,39.959556,27.085856,0.155379,136.001713,93.063592


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,5.096935,95.530043,...,33.159526,28.069098,0.312366,73.326200,12.261262,32.881371,27.294913,0.138014,95.530043,79.527473
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,5.079493,111.887307,...,33.821712,27.215875,0.284241,92.034140,12.000188,33.821712,26.969546,0.147773,111.887307,95.466819
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,5.028469,195.773397,...,33.278941,33.278941,0.000000,0.000000,23.759322,33.278941,27.600132,0.381511,195.773397,156.648623
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,5.313815,225.389836,...,33.888814,33.888814,0.000000,0.000000,23.691455,33.888814,27.224474,0.667380,225.389836,183.307771
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,11.755116,149.206450,...,33.771250,29.688823,5.867400,66.312809,10.007950,33.771250,24.469662,1.240480,149.206450,108.088079
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,12.511319,182.263510,...,34.989055,30.877860,5.867400,66.780750,10.008669,34.989055,23.918692,1.995837,182.263510,128.647991
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,5.030727,219.390853,...,40.385228,40.385228,0.000000,0.000000,23.880049,40.385228,34.241805,0.383408,219.390853,170.277363
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,12.810758,241.868112,...,35.843477,32.158253,5.867400,24.331311,18.311492,35.843477,27.914413,2.296391,241.868112,168.555730
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,4.920243,0.000000,...,42.299130,36.823954,0.271859,151.307410,0.192018,42.299130,42.299130,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,4.204095,119.041564,...,39.959556,29.958103,1.791859,136.652312,6.227499,39.959556,26.671981,0.155379,119.041564,96.056359


In [4]:
# Export resutlts
for model_type, results_df in zip(["data", "physical"], results_dfs):
    
    results_df.loc[results_df["Qdc"] < 1, "Tdc_out"] = np.nan
    results_df.loc[results_df["Qwct"] < 1, "Twct_out"] = np.nan
    
    results_df.to_csv(data_path / f"cc_out_mod_{model_type}.csv", index=False)
    print(f"Exported {data_path / f'cc_out_mod_{model_type}.csv'}")

df.loc[results_dfs[-1]["Qdc"] < 1, "Tdc_out"] = np.nan    
df.loc[results_dfs[-1]["Qwct"] < 1, "Twct_out"] = np.nan

# df.to_csv(data_path / "cc_out_exp.csv")


Exported ../data/cc_out_mod_data.csv
Exported ../data/cc_out_mod_physical.csv


### Take from pre-evaluated

In [9]:
model_type = "physical"

df = pd.read_csv(data_path / "cc_out_exp.csv",)
results_df = pd.read_csv(data_path / f"cc_out_mod_{model_type}.csv")

# Copy some values to the experimental data so they can be used in the visualization
df["Qdc"]  = results_df["Qdc"]
df["Qwct"] = results_df["Qwct"]
df["Qc"]   = results_df["Qc_released"]

display(df)
display(results_df)


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,Twct_out,wdc,wwct,qdc,qwct,Rp,Rs,Qdc,Qwct,Qc
0,19-Feb-2025,36.261009,17.671711,43.180898,23.998493,28.121629,0.450381,3.653911,35.805044,33.494083,...,27.778468,31.974352,22.470002,12.417720,12.259674,0.483,0.054,73.326200,79.527473,150.314103
1,10-Feb-2025,37.774713,13.531843,61.437262,24.000376,27.700450,0.432014,3.442630,37.052174,34.112194,...,27.571514,30.637523,23.770823,12.005001,11.779023,0.500,0.000,92.034140,95.466819,185.227531
2,12-Feb-2025,36.241372,19.685473,23.739778,23.999315,28.145370,0.381511,4.625762,35.705996,21.977082,...,27.961559,0.000000,39.757368,0.235675,23.694971,0.990,0.000,0.000000,156.648623,145.814074
3,19-Feb-2025,37.575024,19.473600,38.711528,23.998248,27.738758,0.667380,4.861102,36.284572,31.971167,...,27.584572,0.000000,50.849622,0.390683,23.691506,0.984,0.201,0.000000,183.307771,176.537254
4,08-Jul-2025,37.389592,27.213752,50.149914,23.999880,28.115154,7.107880,10.170240,36.956869,34.384180,...,24.924979,100.000000,66.439999,13.998713,9.773220,0.417,0.000,66.312809,108.088079,173.279537
5,08-Jul-2025,38.576245,28.395690,42.827841,24.001603,28.341443,7.863237,10.932670,38.064432,35.242784,...,24.004598,100.000000,81.709604,14.000688,9.770016,0.417,0.000,66.780750,128.647991,180.880465
6,09-Jul-2025,42.399588,27.275200,62.124991,24.000049,35.196531,0.383408,4.809472,42.201587,25.526867,...,35.006526,0.000000,39.846699,0.119232,23.694192,0.995,0.000,0.000000,170.277363,133.895591
7,09-Jul-2025,39.453322,31.500935,41.542744,23.999335,28.792534,8.163791,11.842760,38.904821,35.742446,...,27.996002,100.000000,86.901059,5.695932,18.024022,0.763,0.000,24.331311,168.555730,187.774619
8,13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,...,15.001290,30.000000,0.000000,23.798313,0.000540,0.008,0.000,151.307410,0.000000,142.427062
9,02-Dec-2024,43.324459,20.923333,42.056077,17.998552,30.498067,1.947239,3.261430,42.871410,40.086938,...,29.973642,59.506664,24.675237,11.778898,6.036892,0.346,0.000,136.652312,96.056359,202.144133


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,5.096935,95.530043,...,33.159526,28.069098,0.312366,73.326200,12.261262,32.881371,27.294913,0.138014,95.530043,79.527473
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,5.079493,111.887307,...,33.821712,27.215875,0.284241,92.034140,12.000188,33.821712,26.969546,0.147773,111.887307,95.466819
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,5.028469,195.773397,...,33.278941,NaN,0.000000,0.000000,23.759322,33.278941,27.600132,0.381511,195.773397,156.648623
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,5.313815,225.389836,...,33.888814,NaN,0.000000,0.000000,23.691455,33.888814,27.224474,0.667380,225.389836,183.307771
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,11.755116,149.206450,...,33.771250,29.688823,5.867400,66.312809,10.007950,33.771250,24.469662,1.240480,149.206450,108.088079
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,12.511319,182.263510,...,34.989055,30.877860,5.867400,66.780750,10.008669,34.989055,23.918692,1.995837,182.263510,128.647991
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,5.030727,219.390853,...,40.385228,NaN,0.000000,0.000000,23.880049,40.385228,34.241805,0.383408,219.390853,170.277363
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,12.810758,241.868112,...,35.843477,32.158253,5.867400,24.331311,18.311492,35.843477,27.914413,2.296391,241.868112,168.555730
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,4.920243,0.000000,...,42.299130,36.823954,0.271859,151.307410,0.192018,42.299130,NaN,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,4.204095,119.041564,...,39.959556,29.958103,1.791859,136.652312,6.227499,39.959556,26.671981,0.155379,119.041564,96.056359


In [10]:
from phd_visualizations.super_makers import SuperMarker

fig = regression_plot(
    df_ref= df, 
    df_mod= results_df, 
    alternative_labels=f"{model_type.capitalize()}", # "Data", 
    var_ids=["Tdc_out", "Twct_out", "Tc_in", "Tc_out", "Cw"], 
    units=["℃"]*4 + ["l/h"], 
    instruments=[SupportedInstruments.pt100]*4 + [SupportedInstruments.paddle_wheel_flow_meter],
    show_error_metrics=["r2", "mae"],
    var_labels=[
        '<span style="color:#83b366; font-weight:bold;">DC</span> out. temp.', 
        '<span style="color:#9573a6; font-weight:bold;">WCT</span> out. temp.', 
        "Condenser inlet temp.", 
        "Condenser out. temp.", 
        "Water consumption"
    ],
    legend_pos="top_spaced",
    inline_error_metrics_text=True,
    
    # kwargs for layout
    # title_text='<span style="color:#83b366; font-weight:bold;">Combined</span> <span style="color:#9573a6; font-weight:bold;">cooling</span> model validation',
    title_text=f'<span style="font-weight:bold;">Combined</span> <span style="font-weight:bold;">cooler</span> {model_type} model validation',
    title_font_size=21,
    legend_y = 1.015,
    title_y=0.995,
    # legend_font=dict(size=12),
    width=480,
    height=1900,
    vertical_spacing=0.05,
    margin=dict(l=10, r=30, t=120, b=10),
    font_size=10,
    
    # super_marker=SuperMarker(
    #     size_var_id="Qc",
    #     size_var_range=[df["Qc"].min(), df["Qc"].max()],
    #     size_label="Cooling power",
        
    #     fill_var_id="Tamb",
    #     fill_label="Ambient temperature",
    #     fill_var_range=[10, 35],
        
    #     ring_vars_ids=["Qdc", "Qwct"],
    #     ring_labels=("Q̇<sub>dc</sub>", "Q̇<sub>wct</sub>"),
        
    #     show_markers_index=False,
    # )
)

fig


In [ ]:
save_figure(
    fig,
    figure_name=f"cc_model_regression_{model_type}",
    figure_path=Path("../results"),
    formats=["png", "svg"],
    scale=2
)


2025-08-09 13:15:37.342 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/cc_model_regression.png
2025-08-09 13:15:37.948 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/cc_model_regression.svg
